[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/04_weight.ipynb)

# 04 · Document weighting — shares that do not drown in long texts

If we pool every phrase equally, a single long document dominates the comparison. Multilogue therefore averages **within documents first**, then across documents. Uncertainty comes from a **cluster bootstrap** that resamples documents, not phrases.

**Why document level?** Public dialogue corpora are uneven: one consultation response may be a paragraph, another a short essay. The scientific object is usually the document (or conversation), so the weight should be too.

Binary document presence ("did this document raise cluster *c* at all?") is available for temporal or rarity views; this notebook focuses on share comparisons between groups. Distances between whole topic *profiles* (for example earth-mover style comparisons of document mixtures) are left to study packages — multilogue stops at the shares and bootstrap intervals that those distances would consume.

In [ ]:
#@title Setup (run first)
# Free path: synthetic units + FakeClient (no API key).
# Optional: load("polis", ...) downloads public open data into a local cache.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from IPython.display import display, HTML, Markdown
from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


In [ ]:
from multilogue.access import load
from multilogue.dummy import FakeClient
from multilogue.extract import make_extract_work, phrase_table
from multilogue.llm import StageRunner
from multilogue.cluster import (
    Embedder, frozen_or_new_assignments, make_lens_work, resolve_lenses,
    make_label_work, example_phrases_for_clusters,
)
from multilogue.weight import doc_profile, group_compare
from multilogue import viz
import numpy as np

units = load("synthetic", n_docs_per_group=4, units_per_doc=3)["units"]
units_by_id = {u["unit_id"]: u for u in units}
client = FakeClient()
root = FIG.parent
runner = StageRunner(root / "ckpt_w", root / "logs_w", max_workers=2, sleep=lambda _: None)
extraction = {}
for kind in ("concern", "benefit"):
    work = make_extract_work(client, kind=kind, system="e",
        user_for_unit=lambda u, k=kind: u["text"], model="fake-small",
        sentinel=f"NO_{kind.upper()}")
    extraction[kind] = runner.run(f"ex_{kind}", units, lambda u: u["unit_id"], work,
                                  f"ex|{kind}", quiet=True)
phrases = phrase_table(extraction, units_by_id, kinds=["concern", "benefit"],
                       extra_unit_fields=["group"])
k = 4
E = np.asarray(Embedder(root / "emb", client, "fake-emb")(phrases.phrase.tolist()), float)
labels, centroids, _ = frozen_or_new_assignments(phrases, E, k, 0, root / "cl")
phrases = phrases.copy()
phrases["cluster"] = labels
phrases["centroid_sim"] = (E * centroids[labels]).sum(1)
cluster_units = example_phrases_for_clusters(phrases, k, 0)
lab = runner.run("lab", cluster_units, lambda cu: cu["key"],
    make_label_work(client, system="l", prompt_template="Label:\n<<PHRASES>>", model="fake-small"),
    "lab", quiet=True)
names = {cu["cluster"]: lab.get(cu["key"], {}).get("label", f"c{cu['cluster']}") for cu in cluster_units}
cl_desc = "\n".join(f"{c}: {names[c]}" for c in range(k))
lens_runs = runner.run("lens", [1], lambda r: str(r),
    make_lens_work(client, system="L", prompt_template="Group:\n<<CLUSTERS>>",
                   cluster_description=cl_desc, model="fake-small", k=k),
    "lens", quiet=True)
mapping, _ = resolve_lenses(k, lens_runs)
phrases["lens"] = phrases["cluster"].map(mapping)

prof = doc_profile(phrases, "lens", sorted(phrases.lens.unique()))
display(prof.head())
rng = np.random.default_rng(0)
cmp = group_compare(phrases, "group", "lens", None, rng, reps=400,
                    groups=sorted(phrases.group.unique()))
display(cmp)
g0, g1 = sorted(phrases.group.unique())[:2]
viz.group_compare_dumbbell(cmp, label_col="lens", group_a=g0, group_b=g1,
                           path=FIG / "04_group_compare.png",
                           title="Document-weighted lens shares")